# Flausch -- Subtask 2: first span detector then span classifier

In [1]:
# auf Google Colab muss numpy Version runtergesetzt werden, und anschließend Sitzung neu gestartet werden,
# damit Task 1 läuft. Sonst kommt es im Training zu Fehlerabbrüchen
!pip install numpy==1.26.4


In [3]:
# mount drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import os
import sys
import numpy as np

# work with the train data only /content/drive/MyDrive/FlauschData/train_task1.json
path = "/content/drive/MyDrive/FlauschData/"
#path = "Input/Data/train/"
data = pd.read_json(path + "train_task1.json")
dataflausch = data[['document', 'comment_id', 'comment', 'flausch',  'spans', 'span_pairs',
       'types', 'id']].copy()
data2 = pd.read_json(path + "train_task2.json")
data2 = data2[['document', 'comment_id', 'type', 'start', 'end', 'comment', 'flausch', 'span', 'id']].copy()

checkpoint = "deepset/gbert-large"



## Step 1: Train span detector

In [65]:
# generate BIO tag scheme (begin/inside/outside)
# dictionaries to map between tag ids and tag names
tags = ["O","B-span","I-span" ]
tag2id = {name: i for i, name in enumerate(tags)}
id2tag = {i: name for i, name in enumerate(tags)}




In [66]:
tags, tag2id, id2tag

(['O', 'B-span', 'I-span'],
 {'O': 0, 'B-span': 1, 'I-span': 2},
 {0: 'O', 1: 'B-span', 2: 'I-span'})

In [67]:
# set checkpoint and load tokenizer
from transformers import AutoTokenizer
model_name = checkpoint.split("/")[-1] +"_non_labeled_spans"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

In [68]:
#some testing
tokenized = tokenizer(["Das ist ein Horrorfilm!"], return_offsets_mapping=True)
print(tokenized.encodings[0])
print(tokenized.encodings[0].tokens)
print(tokenized.encodings[0].ids)
print(tokenized.encodings[0].offsets)


Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
['[CLS]', 'Das', 'ist', 'ein', 'Horror', '##film', '!', '[SEP]']
[102, 347, 215, 143, 28816, 5241, 3330, 103]
[(0, 0), (0, 3), (4, 7), (8, 11), (12, 18), (18, 22), (22, 23), (0, 0)]


In [69]:
# main function that creates target token sequences for tokenized training data

def align_labels_with_tokens(tokenizer, data, tag2id):
    # tokenize text and get offset_mapping
    text = data["comment"]
    span_pairs = data["span_pairs"]  # list of boundaries of labeled flausch spans
    tokenized_input = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=512)
    # encodings[0] für den einzelnen Text
    encoding = tokenized_input.encodings[0]

    token_ids = encoding.ids  # Token-IDs
    label_ids = [tag2id['O'] for i in  range(len(encoding.tokens))]  # Initialisiere mit O-Labels für jedes Wort/Token


    # Iteriere über jeden Span, der gelabelt werden soll
    for i in range(len(span_pairs)):
        span_start_char = span_pairs[i][0]
        span_end_char = span_pairs[i][1]

        # Finde die Token-Indizes, die diesen Span abdecken
        token_start_idx = encoding.char_to_token(span_start_char)
        token_end_idx = encoding.char_to_token(span_end_char - 1) # end_char ist exklusiv

        # Wenn der Span nicht durch die Tokenisierung abgedeckt wird (z.B. wegen Truncation)
        if token_start_idx is None or token_end_idx is None:
            continue

        # Wenn der Span über mehrere Tokens geht oder ein einzelnes Token ist
        for current_token_idx in range(token_start_idx, token_end_idx + 1):
            if current_token_idx == token_start_idx:
                # Dies ist das erste Token, das den Span abdeckt -> B-Tag
                label_ids[current_token_idx] = tag2id["B-span"]
#            elif current_token_idx == token_end_idx:
#                # Dies ist das letzte Token, das den Span abdeckt -> E-Tag
#                label_ids[current_token_idx] = tag2id["E-span"]
            else:
                # Alle anderen Tokens, die denselben Span abdecken -> I-Tag
                label_ids[current_token_idx] = tag2id["I-span"]

    return tokenized_input["input_ids"], label_ids



In [70]:
i=31
input_ids, label_ids = align_labels_with_tokens(tokenizer, dataflausch.loc[i], tag2id)
ex_spans = dataflausch["spans"][i]

print(ex_spans)
for idx,j in enumerate(label_ids):
    if j != 0:
        print(id2tag[j], tokenizer.decode(input_ids[idx]))

['ich finds gar nicht so schlecht :)']
B-span ich
I-span find
I-span ##s
I-span gar
I-span nicht
I-span so
I-span schlecht
I-span :
I-span )


In [71]:
import pandas as pd
from datasets import Dataset, Features, Value, ClassLabel, Sequence
from sklearn.model_selection import train_test_split # Für den Split in Train/Test



all_input_ids = []
all_labels = []

print("Beginne mit der Vorverarbeitung des DataFrames...")
for idx, row in dataflausch.iterrows():
    input_ids_for_text, labels_for_text = align_labels_with_tokens(tokenizer, row, tag2id)

    # Sammle die Ergebnisse
    all_input_ids.append(input_ids_for_text)
    all_labels.append(labels_for_text)

    if (idx + 1) % 5000 == 0: # Fortschrittsanzeige alle XX Texte
        print(f"Verarbeitete {idx + 1}/{len(dataflausch)} Texte.")

print("Alle Texte erfolgreich vorverarbeitet.")
print(f"Anzahl vorbereiteter Beispiele: {len(all_input_ids)}")

max_label_id = max(id2tag.keys())
id2tag_ordered_list = [None] * (max_label_id + 1) # Erstelle eine Liste der richtigen Größe
for k, v in id2tag.items():
    id2tag_ordered_list[k] = v

features = Features({
    'input_ids': Sequence(Value('int32')), # Sequenz von Ganzzahlen für Token-IDs
    'labels': Sequence(ClassLabel(names=id2tag_ordered_list)) # Sequenz von ClassLabels für NER-Tags
})

# Erstelle ein Dictionary, das die Daten enthält
dataset_dict = {
    'input_ids': all_input_ids,
    'labels': all_labels
}

# Erstelle ein Dataset aus dem Dictionary und den definierten Features
prepared_dataset = Dataset.from_dict(dataset_dict, features=features)

print("\nHugging Face Dataset erfolgreich erstellt.")
print(prepared_dataset)
print("\nBeispiel des ersten Eintrags im Dataset:")
print(prepared_dataset[0])

# Überprüfe den ersten Eintrag noch einmal visuell
first_text = dataflausch["comment"].iloc[0] # .iloc[0] für den ersten Eintrag im DF
first_input_ids = prepared_dataset[0]['input_ids']
first_labels_ids = prepared_dataset[0]['labels']

print(f"\nOriginaltext (erster Eintrag): '{first_text}'")
print(f"Token (decoded): {[tokenizer.decode(t) for t in first_input_ids]}")
print(f"Labels (decoded): {[id2tag[l] for l in first_labels_ids]}")


# Teile das Dataset in Trainings- und Devsets auf (85% Training, 15% Development)
# Remember the current dataset is 90% of the original one. 10% test data have been already set aside
split_dataset = prepared_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset['train']
dev_dataset = split_dataset['test']

# create one datasetdict for train, dev and test
from datasets import DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'dev': dev_dataset,
})

print(f"\nTrainingsdatensatzgröße: {len(train_dataset)}")
print(f"Entwicklungsdatensatzgröße: {len(dev_dataset)}")



Beginne mit der Vorverarbeitung des DataFrames...
Verarbeitete 20000/33351 Texte.
Verarbeitete 5000/33351 Texte.
Verarbeitete 25000/33351 Texte.
Verarbeitete 10000/33351 Texte.
Verarbeitete 15000/33351 Texte.
Verarbeitete 35000/33351 Texte.
Alle Texte erfolgreich vorverarbeitet.
Anzahl vorbereiteter Beispiele: 33351

Hugging Face Dataset erfolgreich erstellt.
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 33351
})

Beispiel des ersten Eintrags im Dataset:
{'input_ids': [102, 1618, 383, 313, 4213, 684, 1717, 5509, 307, 255, 11505, 2827, 103], 'labels': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

Originaltext (erster Eintrag): 'würde ich auch gerne wenn meine Freunde sie nicht schauen würden'
Token (decoded): ['[CLS]', 'würde', 'ich', 'auch', 'gerne', 'wenn', 'meine', 'Freunde', 'sie', 'nicht', 'schauen', 'würden', '[SEP]']
Labels (decoded): ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Trainingsdatensatzgröße: 28348
Entwicklungsdatensatzgröße: 5003


In [72]:
# check some examples
i=31
print(dataflausch["comment"].iloc[i])
print(prepared_dataset[i]['input_ids'])
print(prepared_dataset[i]['labels'])
print([id2tag[prepared_dataset[i]['labels'][j]] for j in range(len(prepared_dataset[i]['labels']))])



Ich finde dich echt Sympatisch ;) ... Bist echt mutig als Junge solche Videos zu machen :) ... Viel Erfolg ... lG Alexandra
[102, 395, 8385, 1199, 8891, 5245, 9553, 191, 3464, 2530, 566, 566, 566, 5572, 8891, 11642, 214, 276, 6436, 3760, 11441, 205, 1327, 853, 2530, 566, 566, 566, 2998, 2993, 566, 566, 566, 228, 30917, 27596, 103]
[0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0]
['O', 'B-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'O', 'O', 'O', 'B-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'I-span', 'O', 'O', 'O', 'B-span', 'I-span', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [73]:
!pip install seqeval
!pip install evaluate

In [74]:
import evaluate
seqeval = evaluate.load("seqeval")

In [75]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

label_list = list(tag2id.keys())

from seqeval.metrics import f1_score, precision_score, recall_score, classification_report


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Konvertierung der numerischen IDs zurück zu den Label-Strings
    # und Entfernen von Padding-Tokens (-100)
    true_labels = []
    for label_sequence in labels:
        true_labels.append([id2tag[l] for l in label_sequence if l != -100])

    true_predictions = []
    for prediction_sequence, label_sequence in zip(predictions, labels):
        true_predictions.append([
            id2tag[p] for p, l in zip(prediction_sequence, label_sequence) if l != -100
        ])

    # Berechne Metriken mit explizitem zero_division=0
    # Dies ist der Standard von seqeval und sorgt dafür, dass undefinierte Werte 0.0 werden.
    # Es hilft, die UndefinedMetricWarning zu kontrollieren.
    precision = precision_score(true_labels, true_predictions, zero_division=0)
    recall = recall_score(true_labels, true_predictions, zero_division=0)
    f1 = f1_score(true_labels, true_predictions, zero_division=0)

    # Generiere und gib den detaillierten Klassifikationsbericht aus.
    # Das `labels=label_list` Argument stellt sicher, dass alle deine definierten Labels
    # im Bericht erscheinen, auch wenn sie im aktuellen Batch nicht vorkommen.
    report = classification_report(
        true_labels,
        true_predictions,
        digits=4, # Anzahl der Dezimalstellen im Bericht
        zero_division=0, # Steuerung der 0-Division für den Bericht
    )

    print("\n--- Classification Report (Aktueller Evaluationsschritt) ---")
    print(report)
    print("-----------------------------------------------------------")

    return {"precision": precision, "recall": recall, "f1": f1}




In [76]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [77]:
# load model with the correct head for token classification with the number of labels
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

num_labels = len(tag2id)  # Anzahl der Labels

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2tag, label2id=tag2id
)

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [78]:
# huggingface login
from huggingface_hub import login
login()


In [79]:
from transformers import TrainingArguments, Trainer
import os



# Define training parameters
training_args = TrainingArguments(
    output_dir="flausch_span_"+ model_name, # Verzeichnis für Modell, Checkpoints und Logs
    learning_rate=2e-5,
    metric_for_best_model="eval_f1", # competition ranking metric
    greater_is_better=True, # Setze auf False, wenn du Loss minimieren willst
    per_device_train_batch_size=16, # Reduziere um OOM zu vermeiden
    per_device_eval_batch_size=16,  # Reduziere um OOM zu vermeiden
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy= "steps",
    load_best_model_at_end=True,
    logging_steps=750, # Alle XX Schritte loggen
    save_steps = 750,
    fp16=True, # Für schnellere und speichereffizientere Berechnung auf GPU
    gradient_checkpointing=True, # Hilft gegen OOM bei großen Modellen, macht Training langsamer
    save_total_limit=1, # Nur das beste Modell speichern
    report_to="none",
    push_to_hub=False, # Zuerst trainieren, dann manuell pushen
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # Eigene Funktion zur Metrikberechnung
)


/tmp/ipython-input-79-2204816505.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [80]:
model_name

'gbert-large_non_labeled_spans'

In [81]:
print("\nPredictions on dev set before training.")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset_dict["dev"])


Predictions on dev set before training.



--- Classification Report (Aktueller Evaluationsschritt) ---
              precision    recall  f1-score   support

        span     0.0003    0.0028    0.0005      2111

   micro avg     0.0003    0.0028    0.0005      2111
   macro avg     0.0003    0.0028    0.0005      2111
weighted avg     0.0003    0.0028    0.0005      2111

-----------------------------------------------------------


In [82]:

# Training
trainer.train()


Step,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1
750,0.259500,0.175390,0.041100,0.510524,0.631928,0.564776
1500,0.182100,0.168330,0.041100,0.595396,0.637139,0.615561
2250,0.139400,0.164938,0.041100,0.572222,0.683089,0.622760
3000,0.109900,0.154673,0.041100,0.667688,0.722406,0.693970
3750,0.098900,0.150671,0.041100,0.642917,0.730933,0.684106
4500,0.060500,0.171187,0.041100,0.672406,0.739934,0.704556
5250,0.060300,0.173615,0.041100,0.674989,0.742776,0.707262



--- Classification Report (Aktueller Evaluationsschritt) ---
              precision    recall  f1-score   support

        span     0.5105    0.6319    0.5648      2111

   micro avg     0.5105    0.6319    0.5648      2111
   macro avg     0.5105    0.6319    0.5648      2111
weighted avg     0.5105    0.6319    0.5648      2111

-----------------------------------------------------------

--- Classification Report (Aktueller Evaluationsschritt) ---
              precision    recall  f1-score   support

        span     0.5954    0.6371    0.6156      2111

   micro avg     0.5954    0.6371    0.6156      2111
   macro avg     0.5954    0.6371    0.6156      2111
weighted avg     0.5954    0.6371    0.6156      2111

-----------------------------------------------------------

--- Classification Report (Aktueller Evaluationsschritt) ---
              precision    recall  f1-score   support

        span     0.5722    0.6831    0.6228      2111

   micro avg     0.5722    0.6831    0

TrainOutput(global_step=5316, training_loss=0.12927835735725765, metrics={'train_runtime': 2733.5522, 'train_samples_per_second': 31.111, 'train_steps_per_second': 1.945, 'total_flos': 9815327068576728.0, 'train_loss': 0.12927835735725765, 'epoch': 3.0})

In [83]:

print("\nPredictions on dev set after training.")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset_dict["dev"])


Predictions on dev set after training.



--- Classification Report (Aktueller Evaluationsschritt) ---
              precision    recall  f1-score   support

        span     0.6750    0.7428    0.7073      2111

   micro avg     0.6750    0.7428    0.7073      2111
   macro avg     0.6750    0.7428    0.7073      2111
weighted avg     0.6750    0.7428    0.7073      2111

-----------------------------------------------------------


In [84]:
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Wiebke/flausch_span_gbert-large_non_labeled_spans/commit/1c93984d8340c11378f3f816b4226a02f52fd145', commit_message='End of training', commit_description='', oid='1c93984d8340c11378f3f816b4226a02f52fd145', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Wiebke/flausch_span_gbert-large_non_labeled_spans', endpoint='https://huggingface.co', repo_type='model', repo_id='Wiebke/flausch_span_gbert-large_non_labeled_spans'), pr_revision=None, pr_num=None)

## Step 2: Train span classifier

In [10]:
# Variant A (only Flausch spans are input for classifier)
# append all lists in list of spans
spans = [s for sx in dataflausch["spans"] for s in sx]
types = [s for sx in dataflausch["types"] for s in sx]
#df_spanlabel = pd.DataFrame(list(zip(spans, types)), columns=["span", "type"])
#model_name =   checkpoint.split("/")[-1] + "_span_classifier"

# Variant B (Flausch and non-Flausch spans are input for classifier)
df_spanlabel = pd.read_csv(path + "phrases_with_labels_inlcuding_not_flausch_train.csv")
df_spanlabel.rename(columns={"phrase": "span", "label": "type"}, inplace=True)
model_name =   checkpoint.split("/")[-1] + "_span_classifier_with_nonspan"




In [11]:
df_spanlabel

,span,type
0,würde ich auch gerne wenn meine Freunde sie ni...,not
1,Ich bin neun,not
2,und bin ein Mädchen,not
3,", brauche aber nur fünf",not
4,bis zehn,not
...,...,...
69778,geiles viedio,positive feedback
69779,daummen hoch,positive feedback
69780,ich glaube,not
69781,sie meint roman ^^,not


In [12]:
labels = list(df_spanlabel["type"].unique())
id2label = {i: l for i, l in enumerate(labels)}
label2id = {v: k for k, v in id2label.items()}
df_spanlabel['label'] = df_spanlabel['type'].map(label2id)





len(spans), len(types)

(14282, 14282)

In [13]:
from datasets import Dataset, DatasetDict
column = "span"
dataset_spanlabel = Dataset.from_pandas(df_spanlabel)
dataset_spanlabel

Dataset({
    features: ['span', 'type', 'label'],
    num_rows: 69783
})

In [14]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(examples):
    inputs = [str(x) for x in examples[column]]
    # Der Tokenizer gibt hier Listen von Listen zurück (für den Batch)
    tokenized_inputs = tokenizer(inputs, truncation=True, max_length=512)
    return tokenized_inputs



# Tokenize dataset
tokenized_dataset = dataset_spanlabel.map(tokenize_function, batched=True)

keep_cols = ['label', 'input_ids', 'attention_mask']

# Setze das Format auf PyTorch-Tensoren
tokenized_dataset.set_format("torch")
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col not in keep_cols])


Map:   0%|          | 0/69783 [00:00<?, ? examples/s]

In [15]:
# data consists of 90% of the data
# 85% of this data is used as train data and 15% as dev data
train_val_split = tokenized_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = train_val_split["train"]
dev_dataset = train_val_split["test"]


# Create a DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'dev': dev_dataset,
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 59315
    })
    dev: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 10468
    })
})

In [16]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np

# load model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Data Collator
# Padd alle Batches auf die längste Sequenz im Batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# Define metric function
from sklearn.metrics import accuracy_score, f1_score
def compute_classifier_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}


model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at deepset/gbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:

# Define TrainingArguments
from transformers import TrainingArguments, Trainer
import os

training_args = TrainingArguments(
    output_dir="./task2_flausch_classification_" + model_name,
    learning_rate=2e-5,
    per_device_train_batch_size=16, # Reduziere um OOM zu vermeiden
    per_device_eval_batch_size=16,  # Reduziere um OOM zu vermeiden
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    logging_steps=1000, # Alle XX Schritte loggen
    save_steps=1000,
    fp16=True, # Für schnellere und speichereffizientere Berechnung auf GPU
    gradient_checkpointing=True, # Hilft gegen OOM bei großen Modellen, macht Training langsamer
    save_total_limit=1, # Nur das beste Modell speichern
    report_to="none", # Wichtig: W&B und andere Logger deaktivieren, falls nicht gewünscht
    push_to_hub=False, # Zuerst trainieren, dann manuell pushen
    metric_for_best_model="f1", # competition ranking metric
    greater_is_better=True, # Setze auf False, wenn du Loss minimieren willst
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_classifier_metrics,
)




/tmp/ipython-input-17-1939512889.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [18]:
#
trainer.train()

print("\nTraining abgeschlossen.")


Step,Training Loss,Validation Loss,Accuracy,F1
1000,0.374900,0.309925,0.914883,0.917360
2000,0.280400,0.232953,0.937142,0.930483
3000,0.267200,0.236936,0.939148,0.933927
4000,0.224800,0.229488,0.942682,0.940615
5000,0.194200,0.224430,0.945166,0.942238
6000,0.187300,0.230992,0.942300,0.939277
7000,0.176800,0.215486,0.946886,0.945018
8000,0.144300,0.229526,0.945357,0.944902
9000,0.119800,0.229501,0.948414,0.947376
10000,0.123800,0.227770,0.947937,0.946877



Training abgeschlossen.


In [19]:
print("\nPredictions on dev set after training.")

predictions_output = trainer.predict(test_dataset=dataset_dict["dev"])
print(predictions_output.metrics)


Predictions on dev set after training.


{'test_loss': 0.22950111329555511, 'test_accuracy': 0.9484142147497134, 'test_f1': 0.9473763127618265, 'test_runtime': 20.954, 'test_samples_per_second': 499.572, 'test_steps_per_second': 31.259}


In [20]:
from huggingface_hub import login
login()

In [22]:
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan/commit/a1f905324ecbce4e0020dcb506feb8b09585da09', commit_message='End of training', commit_description='', oid='a1f905324ecbce4e0020dcb506feb8b09585da09', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan', endpoint='https://huggingface.co', repo_type='model', repo_id='Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan'), pr_revision=None, pr_num=None)

In [37]:
list(dataset_dict["dev"]["label"])

[tensor(6),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(3),
 tensor(0),
 tensor(3),
 tensor(3),
 tensor(0),
 tensor(0),
 tensor(2),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(4),
 tensor(0),
 tensor(0),
 tensor(3),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(1),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(4),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(1),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(3),
 tensor(3),
 tensor(0),
 tensor(0),
 tensor(6),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(1),
 tensor(1),
 tensor(0),
 tensor(3),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(1),
 tensor(0),
 tensor(0),
 tensor(4),
 tensor(3),
 tensor(4),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 tensor(0),
 ten

In [38]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = [id2label[x] for x in list(predictions_output.label_ids)]
y_true = [id2label[int(x)] for x in list(dataset_dict["dev"]["label"])]
print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

                       precision    recall  f1-score   support

affection declaration       1.00      1.00      1.00       366
            agreement       1.00      1.00      1.00        39
            ambiguous       1.00      1.00      1.00        28
           compliment       1.00      1.00      1.00       361
        encouragement       1.00      1.00      1.00        97
            gratitude       1.00      1.00      1.00        30
     group membership       1.00      1.00      1.00        21
             implicit       1.00      1.00      1.00        29
                  not       1.00      1.00      1.00      8371
    positive feedback       1.00      1.00      1.00      1116
             sympathy       1.00      1.00      1.00        10

             accuracy                           1.00     10468
            macro avg       1.00      1.00      1.00     10468
         weighted avg       1.00      1.00      1.00     10468

[[ 366    0    0    0    0    0    0    0    0    0 

In [ ]:
# test that size and content of y_pred is correct

In [ ]:
# show ith prediction
i = 17
print(id2label[predictions_output.label_ids[i]])
print(tokenizer.decode(dataset_dict["dev"][i]["input_ids"]))
print(id2label[int(dataset_dict["dev"][i]["label"])])



## Putting it together: span detection and classification for task 2

In [95]:
detector = checkpoint.split("/")[-1] +"_non_labeled_spans"
classifier = checkpoint.split("/")[-1] + "_span_classifier"



In [99]:
from transformers import pipeline

pipe = pipeline("text-classification", model="Wiebke/results_flausch_classification_gbert-large_span_classifier")



config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/729k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


In [101]:
pipe(["Ihr seid die Größten!!!","macht weiter so!","U2 forever","das Auto ist rot"])

[{'label': 'affection declaration', 'score': 0.997929573059082},
 {'label': 'encouragement', 'score': 0.9986856579780579},
 {'label': 'group membership', 'score': 0.7810184955596924},
 {'label': 'positive feedback', 'score': 0.9801151752471924}]